# Copyright

<PRE>
Jelen iPython notebook a Budapesti Műszaki és Gazdaságtudományi Egyetemen tartott
"MI ágensek és multiágensrendszerek fejlesztése" tantárgy segédanyagaként készült.

A notebook bármely részének újra felhasználása, publikálása csak a szerzők írásos beleegyezése esetén megegengedett.

2026 (c) Potyók Csaba (potyok kukac mit pont bme pont hu)
</PRE>

# LLM ágensek multiágens környezetben

A gyakorlat során az A2A (Agent to Agent) protokoll alkalmazásával készítünk egy többágenses rendszert. A gyakorlat célja az ágensek közötti együttműködés bemutatása egy, a korábbi gyakorlatok során már bemutatott példán keresztül.

# Szükséges könyvtárak

A gyakorlat során a [Pydantic AI](https://ai.pydantic.dev/) könyvtárat használjuk, mivel ennek segítségével típusbiztosan és egyszerűen tudunk eszközöket definiálni.

In [ ]:
# Telepítsd a szükséges könyvtárakat
# Ollama Cloud használatához szükséges csomagok
!pip install pydantic-ai sqlmodel "pydantic-ai-slim[fastmcp]" a2a-sdk==0.3.25 requests --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.6/149.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.8/894.8 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.3/92.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.6/738.6 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.7/831.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Környezeti változók beállítása

Ollama Cloud API kulcs beállítása. A Cloud verzióhoz szükséges API kulcsot a https://ollama.com/profile/api-keys címen szerezheti be.

In [ ]:
import os

# Ollama Cloud API kulcs beállítása
# A kulcsot a https://ollama.com/profile/api-keys címen szerezheti be
os.environ["OLLAMA_API_KEY"] = "your-ollama-cloud-api-key-here"
os.environ["OLLAMA_BASE_URL"] = "https://ollama.com/v1"

# Ollama alapértelmezett URL-je
os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"

print("Ollama beállítások:")
print(f"  OLLAMA_BASE_URL: {os.environ.get('OLLAMA_BASE_URL', 'Nincs beállítva')}")

# A2A - Együttműködés

Ebben a szakaszban megvalósítjuk a korábban már elkészített `calendar_agent` és `movie_agent` ágenseinket, mint A2A ágensek az `a2a-sdk` segítségével. A többágenses rendszerünkben egy `program_plan_agent` ágens a megvalósított A2A ágenseinkkel együttműködve fog programokat beütemezni a felhasználó naptárába.

In [3]:
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentSkill,
    Message,
    Part,
    Role,
    SendMessageRequest,
    MessageSendParams,
    TextPart,
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
)
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore

from a2a.client import A2ACardResolver, A2AClient
from a2a.client.client import ClientConfig
from a2a.client.client_factory import ClientFactory

from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.utils.artifact import new_text_artifact, new_data_artifact
from a2a.utils.message import new_agent_text_message
from a2a.utils.task import new_task

import httpx
from uuid import uuid4
import uvicorn
import threading
from pydantic import BaseModel
from typing import List, Dict
import asyncio
import nest_asyncio
from sqlmodel import SQLModel, Field, create_engine, Session, select, JSON, Column, Relationship, or_, and_
from typing import List, Optional, Literal, Union
from datetime import datetime, timedelta

import logfire

from pydantic_ai import (
    Agent,
    RunContext,
    UsageLimits,
    ModelMessage,
    RunUsage
)
from fastmcp import FastMCP
from pydantic_ai.toolsets.fastmcp import FastMCPToolset

Annak érdekében, hogy egymásba ágyazott aszinkron hívásokat lehessen indítani engedélyezzük ezt az alábbi kóddal.

In [4]:
nest_asyncio.apply()

In [ ]:
logfire.configure(send_to_logfire='if-token-present')
logfire.instrument_pydantic_ai()

# Ollama Cloud kapcsolat ellenőrzése
import requests

def check_ollama_connection():
    """Ollama Cloud kapcsolat ellenőrzése"""
    api_key = os.environ.get("OLLAMA_API_KEY", "")
    base_url = os.environ.get("OLLAMA_BASE_URL", "")
    
    if api_key == "your-ollama-cloud-api-key-here":
        print("⚠️ Figyelmeztetés: Kérlek, állítsd be az Ollama Cloud API kulcsot!")
        print("   Töltsd le a kulcsot a https://ollama.com/profile/api-keys címen")
        print("   és cseréld le a 'your-ollama-cloud-api-key-here' szöveget a beállítás cellában.")
        return False
    
    try:
        headers = {"Authorization": f"Bearer {api_key}"}
        response = requests.get(f"{base_url}/models", headers=headers, timeout=10)
        if response.status_code == 200:
            print("✅ Ollama Cloud kapcsolat sikeres!")
            models = response.json().get('data', [])
            if models:
                print("Elérhető modellek:")
                for model in models:
                    model_id = model.get('id', model.get('name', 'ismeretlen'))
                    print(f"  - {model_id}")
            else:
                print("⚠️ Nem találhatók modellek. Kérlek, ellenőrizd az API kulcsodat.")
            return True
        else:
            print(f"❌ Ollama Cloud kapcsolati hiba: {response.status_code}")
            print(f"   Válasz: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Ollama Cloud nem érhető el: {e}")
        print("Kérlek, ellenőrizd az API kulcsodat és az internetkapcsolatot.")
        return False

check_ollama_connection()

Kiválasztunk egy olyan modellt, amely képes eszközhasználatra.

In [ ]:
model_name = 'ollama:llama3.2'

## 1. Naptárkezelő ágens

Definiáljuk a `CalendarEntry` adatmodellt, amely tartalmazza a következő mezőket: `title`, `description`, `start_time` és `end_time`. Adjunk meg minden mezőhöz egy rövid leírást. Az időpontok kezeléséhez használjuk a `datetime` modult.

In [7]:
class CalendarEntry(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str = Field(description="The title of the event")
    description: str = Field(description="The description of the event")
    start_time: datetime = Field(description="The start time of the event")
    end_time: datetime = Field(description="The end time of the event")

In [8]:
calendar_sqlite_url = "sqlite:///calendar.db"
calendar_engine = create_engine(calendar_sqlite_url)

In [9]:
SQLModel.metadata.create_all(calendar_engine)

Definiáljuk a `Calendar` nevű MCP szervert, továbbá adjuk hozzá az `insert_event`eszközt, amelynek segítségével programokat/eseményeket lehet beszúrni a naptárba, valamint a `get_today` eszközt, amely segítségével a mai dátumot lehet lekérdezni!

In [10]:
calendar_mcp_server = FastMCP('Calendar')

Definiáljuk a `get_today` eszközt, amelynek a segítségével a mai dátumot lehet lekérdezni!

In [11]:
@calendar_mcp_server.tool()
def get_today():
  """Returns today's date."""
  return datetime.now().date()

Definiáljuk a `insert_event` eszközt, amivel az ágens fel tud venni egy új naptárbejegyzést!

In [12]:
@calendar_mcp_server.tool()
def insert_event(title: str, description: str, start_time: datetime, end_time: datetime):
  """Insert an event to the calendar."""
  with Session(calendar_engine) as session:
    new_event = CalendarEntry(
      title=title,
      description=description,
      start_time=start_time,
      end_time=end_time
    )
    session.add(new_event)
    session.commit()

Definiáljuk a `Calendar` nevű MCP szervert, továbbá adjuk hozzá a következő eszközöket, amelyeket az ágensünket fog majd használni!

In [13]:
toolset = FastMCPToolset(calendar_mcp_server)

/tmp/ipykernel_11679/2037691365.py:1: DeprecationWarning: `FastMCPToolset` is deprecated and will be removed in v2. Use `pydantic_ai.mcp.MCPToolset` instead — it is also built on the FastMCP `Client` and accepts a pre-built `fastmcp.Client` or any input FastMCP can build a transport from, while adding full parity with the legacy `MCPServer*` classes (caching, resource methods, sampling shortcuts, OAuth auth). See the migration guide in the v2 release notes.
  toolset = FastMCPToolset(calendar_mcp_server)


Definiáljuk a `calendar_agent` ágenst!

In [14]:
calendar_agent = Agent(
    model_name,
    system_prompt=(
        "You are a calendar management agent whose job is to answer questions asked by users based on calendar entries. "
        "Check the calendar for today's date."
    ),
    toolsets=[toolset]
)

/usr/local/lib/python3.12/dist-packages/pydantic_ai/agent/__init__.py:409: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


Definiáljuk az ágensünk képességeit (`AgentSkill`)!

In [15]:
get_today_skill = AgentSkill(
    id='get_today',
    name='Get today\'s date',
    description='Returns date of today',
    tags=['today', 'date'],
    examples=['What day is it today?'],
)

insert_event_skill = AgentSkill(
    id='insert_event',
    name='Insert an event',
    description='Insert an event to the calendar',
    tags=['event', 'insert', 'calendar'],
    examples=['Enter the one-hour yoga class starting at 8 p.m. in your calendar for tomorrow!', 'Add a two-hour meeting with the team today at 2:00 p.m.'],
)

Ezután adjuk meg az `AgentCard`-unkat, amely leírja az ágensünket!

In [16]:
calendar_agent_card = AgentCard(
    name='calendar_agent',
    url="http://localhost:10020/",
    version="1.0.0",
    description='Manages calendar events.',
    skills=[get_today_skill, insert_event_skill],
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=AgentCapabilities(
        streaming=True
    ),
    supported_interfaces=[
        AgentInterface(
            transport='JSONRPC',
            url='http://localhost:10020',
        )
    ],
)

Az A2A szerverhez először egy `AgentExecutor`-t kell készítenünk, amely kezeli majd az ágens üzeneteit. Legyen ez a `CalendarAgentExecutor`, amely használja fel a korábban definiált `calendar_agent`-t.

> Fontos! A jelenlegi Pydantic AI verzióban bizonyos mezőket az A2A protokollból szigorúban vár el, mint a hivatalos SDK, így előfordulhat, hogy ezeket a helyzeteket orvosolni szükséges.

In [17]:
class CalendarAgentExecutor(AgentExecutor):
    """Calendar Agent Executor Implementation."""

    def __init__(self) -> None:
        self.agent = calendar_agent

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        """Execute the agent process and enqueue the final response."""
        task = context.current_task or new_task(context.message)
        await event_queue.enqueue_event(task)

        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(
                task_id=context.task_id,
                context_id=context.context_id,
                status=TaskStatus(
                    state=TaskState.working,
                    message=new_agent_text_message('Processing request...'),
                ),
                final=False
            )
        )

        message = context.message
        extracted_text = " ".join([part.root.text for part in message.parts if type(part.root) is TextPart])
        response = await self.agent.run(extracted_text)

        await event_queue.enqueue_event(
            TaskArtifactUpdateEvent(
                task_id=context.task_id,
                context_id=context.context_id,
                artifact=new_text_artifact(name='result', text=response.output),
            )
        )
        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(
                task_id=context.task_id,
                context_id=context.context_id,
                status=TaskStatus(state=TaskState.completed),
                final=True
            )
        )

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        """Raise exception as cancel is not supported."""
        raise Exception('cancel not supported')


Legvégül készítsük el az A2A szerverünket, amely majd várni fogja az ágensünknek küldött üzeneteket!

In [18]:
request_handler = DefaultRequestHandler(
    agent_executor=CalendarAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

server = A2AStarletteApplication(
    agent_card=calendar_agent_card,
    http_handler=request_handler,
)

Indítsuk el egy háttérszálon az ágens A2A szerverét annak érdekében, hogy Colabban további kódokat tudjunk majd még futtatni!

In [19]:
def run_calendar_agent():
    uvicorn.run(server.build(), host="127.0.0.1", port=10020, log_level="info")

In [20]:
daemon_thread = threading.Thread(target=run_calendar_agent, daemon=True)
daemon_thread.start()

## 2. Mozi figyelő ágens

In [21]:
from enum import Enum

class MovieGenre(str, Enum):
  thriller = 'thriller'
  action = 'action'
  adventure = 'adventure'
  drama = 'drama'
  sci_fi = 'sci-fi'
  romantic = 'romantic'
  comedy = 'comedy'

Definiáljuk a `Movie` és a `MovieScreening` adatmodelleket! A `Movie` egy mozifilmet reprezentál, amelynek a következő mezői vannak: `title`, `short_description`, `genre` és `length`. A `MovieScreening` a vetítésekhez tartozó információkat írja le, ehhez `movie_id` és `start_time` mezőket definiál.

In [22]:
class Movie(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str = Field(description="The title of the movie")
    short_description: str = Field(description="The short description of the movie")
    genre: List[MovieGenre] = Field(description="The list of genre of the film", sa_column=Column(JSON))
    length: int = Field(description="The length of the movie in minutes")

    screenings: List["MovieScreening"] = Relationship(back_populates="movie")

class MovieScreening(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    movie_id: int = Field(description="The id of the movie", foreign_key="movie.id")
    start_time: datetime = Field(description="The start time of the screening")

    movie: Optional[Movie] = Relationship(back_populates="screenings")

Hozzunk létre olyan adatmodelleket, amelyek általunk megszabadott módon írják le az egyes filmekhez tartozó információkat! A `MovieInfo` modell csak `id` és `title` mezőket tartalmazza, míg a `DetailedMovieInfo` modell tartalmazza továbbá a `genre`, `length` és `screenings` attributúmokat is.

In [23]:
from pydantic import Field as PydanticField

class MovieInfo(BaseModel):
    id: int = PydanticField(description="The id of the movie")
    title: str = PydanticField(description="The title of the movie")

class DetailedMovieInfo(MovieInfo):
    short_description: str = PydanticField(description="The short description of the movie")
    genre: List[MovieGenre]
    length: int = PydanticField(description="The length of the movie in minutes")
    screenings: List[datetime] = PydanticField(description="The start time of screenings")

Definiáljuk az adatbázist ahol tárolni fogjuk ezeket az információkat!

In [24]:
movie_sqlite_url = "sqlite:///movie.db"
movie_engine = create_engine(movie_sqlite_url)

In [25]:
SQLModel.metadata.create_all(movie_engine)

### Példa adatok

Adjunk meg néhány példa filmet és vetítési alkalmat, amelyet az adatbázisunk fog tartalmazni!

In [26]:
with Session(movie_engine) as session:
    today = datetime.now()
    tomorrow = today + timedelta(days=1)
    example_movies = [
        Movie(
            title="Interstellar",
            short_description="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
            genre=["sci-fi", "adventure"],
            length=169,
            screenings=[
                MovieScreening(start_time=today.replace(hour=9, minute=0)),
                MovieScreening(start_time=today.replace(hour=18, minute=30)),
                MovieScreening(start_time=today.replace(hour=20, minute=0)),
                MovieScreening(start_time=tomorrow.replace(hour=9, minute=0)),
                MovieScreening(start_time=tomorrow.replace(hour=18, minute=30)),
                MovieScreening(start_time=tomorrow.replace(hour=20, minute=0))
            ]
        ),
        Movie(
            title="The Dark Knight",
            short_description="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham.",
            genre=["action", "thriller"],
            length=152,
            screenings=[
                MovieScreening(start_time=today.replace(hour=10, minute=0)),
                MovieScreening(start_time=today.replace(hour=21, minute=0)),
                MovieScreening(start_time=tomorrow.replace(hour=10, minute=0)),
                MovieScreening(start_time=tomorrow.replace(hour=21, minute=0))
            ]
        ),

    ]
    session.add_all(example_movies)
    session.commit()

### MCP eszközök és ágens elkészítése

Definiáljuk a `Movie` nevű MCP szerverünket, illetve készítsük el hozzá a szükséges eszközöket!

In [27]:
movie_mcp_server = FastMCP('Movie')

Definiáljunk egy `search_movies_by_filter` eszközt, amely összetett lekérdezéseket tesz lehetővé! A `genres` paraméterében opcionálisan megadható egy műfaj lista (`or` művelettel összefűzve a feltételben), illetve a `target_date` paraméterében opcionálisan megadható dátum.

In [28]:
@movie_mcp_server.tool()
def search_movies_by_filter(
    genres: Optional[List[Literal['thriller', 'action', 'adventure', 'drama', 'sci-fi', 'romantic', 'comedy']]] = None,
    target_date: Optional[datetime] = None
    ):
  """Returns a list of movies that meet the filter criteria."""
  with Session(movie_engine) as session:
    statement = select(Movie)

    if target_date:
      start_of_day = target_date.replace(hour=0, minute=0, second=0, microsecond=0)
      end_of_day = start_of_day + timedelta(days=1)
      statement = statement.where(
          Movie.screenings.any(
              and_(
                    MovieScreening.start_time >= start_of_day,
                    MovieScreening.start_time < end_of_day
                )
            )
      )

    if genres:
      conditions = [Movie.genre.contains(g) for g in genres]
      statement = statement.where(or_(*conditions))

    results = session.exec(statement).all()

    return [DetailedMovieInfo(
        title=movie.title,
        id=movie.id,
        short_description=movie.short_description,
        genre=movie.genre,
        length=movie.length,
        screenings=[screening.start_time for screening in movie.screenings if target_date is None or start_of_day <= screening.start_time < end_of_day]
    ) for movie in results]

Készítsük el a mozi kereső ágensünket!

In [29]:
toolset = FastMCPToolset(movie_mcp_server)

/tmp/ipykernel_11679/2175491956.py:1: DeprecationWarning: `FastMCPToolset` is deprecated and will be removed in v2. Use `pydantic_ai.mcp.MCPToolset` instead — it is also built on the FastMCP `Client` and accepts a pre-built `fastmcp.Client` or any input FastMCP can build a transport from, while adding full parity with the legacy `MCPServer*` classes (caching, resource methods, sampling shortcuts, OAuth auth). See the migration guide in the v2 release notes.
  toolset = FastMCPToolset(movie_mcp_server)


In [30]:
movie_agent = Agent(
    model_name,
    system_prompt=(
        "You are a movie search agent who can provide information about movies that meet the user's request. "
    ),
    toolsets=[toolset]
)

Definiáljuk az ágensünk képességeit (`AgentSkill`)!

In [31]:
search_movie_by_filter_skill = AgentSkill(
    id='search_movie_by_filter',
    name='Search movie by given filter',
    description='Returns a list of movies that meet the filter criteria',
    tags=['movie', 'search', 'filter'],
    examples=['What adventure movies are showing today?', 'What movies are showing tomorrow?'],
)

Ezután adjuk meg az `AgentCard`-unkat, amely leírja az ágensünket!

In [32]:
movie_agent_card = AgentCard(
    name='movie_agent',
    url="http://localhost:10021/",
    version="1.0.0",
    description='Search among movies currently showing in theares.',
    skills=[search_movie_by_filter_skill],
    default_input_modes=['text'],
    default_output_modes=['text'],
    capabilities=AgentCapabilities(
        streaming=True
    ),
    supported_interfaces=[
        AgentInterface(
            transport='JSONRPC',
            url='http://localhost:10021',
        )
    ],
)

Az A2A szerverhez először egy `AgentExecutor`-t kell készítenünk, amely kezeli majd az ágens üzeneteit. Legyen ez a `MovieAgentExecutor`, amely használja fel a korábban definiált `movie_agent`-t.

> Fontos! A jelenlegi Pydantic AI verzióban bizonyos mezőket az A2A protokollból szigorúban vár el, mint a hivatalos SDK, így előfordulhat, hogy ezeket a helyzeteket orvosolni szükséges.

In [33]:
class MovieAgentExecutor(AgentExecutor):
    """Movie Agent Executor Implementation."""

    def __init__(self) -> None:
        self.agent = movie_agent

    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        """Execute the agent process and enqueue the final response."""
        task = context.current_task or new_task(context.message)
        await event_queue.enqueue_event(task)

        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(
                task_id=context.task_id,
                context_id=context.context_id,
                status=TaskStatus(
                    state=TaskState.working,
                    message=new_agent_text_message('Processing request...'),
                ),
                final=False
            )
        )

        message = context.message
        extracted_text = " ".join([part.root.text for part in message.parts if type(part.root) is TextPart])
        response = await self.agent.run(extracted_text)

        await event_queue.enqueue_event(
            TaskArtifactUpdateEvent(
                task_id=context.task_id,
                context_id=context.context_id,
                artifact=new_text_artifact(name='result', text=response.output),
            )
        )
        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(
                task_id=context.task_id,
                context_id=context.context_id,
                status=TaskStatus(state=TaskState.completed),
                final=True
            )
        )

    async def cancel(
        self, context: RequestContext, event_queue: EventQueue
    ) -> None:
        """Raise exception as cancel is not supported."""
        raise Exception('cancel not supported')


Legvégül készítsük el az A2A szerverünket, amely majd várni fogja az ágensünknek küldött üzeneteket!

In [34]:
request_handler = DefaultRequestHandler(
    agent_executor=MovieAgentExecutor(),
    task_store=InMemoryTaskStore(),
)

server = A2AStarletteApplication(
    agent_card=movie_agent_card,
    http_handler=request_handler,
)

Indítsuk el egy háttérszálon az ágens A2A szerverét annak érdekében, hogy Colabban további kódokat tudjunk majd még futtatni!

In [35]:
def run_movie_agent():
    uvicorn.run(server.build(), host="127.0.0.1", port=10021, log_level="info")

In [36]:
daemon_thread = threading.Thread(target=run_movie_agent, daemon=True)
daemon_thread.start()

INFO:     Started server process [11679]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


Csatlakozzunk az A2A klienssel a szerverhez és kérdezzük le az AgentCard-unkat!

In [37]:
from IPython.display import display, JSON, Markdown

In [38]:
base_url = "http://localhost:10020"

Vizsgáljuk meg, hogyan néz ki az elkészített AgentCard-unk!

In [39]:
async with httpx.AsyncClient() as httpx_client:
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=base_url,
    )

    public_card = (
        await resolver.get_agent_card()
    )

    display(JSON(public_card.model_dump_json()))

INFO:     127.0.0.1:40446 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/IPython/core/display.py:911: UserWarning: JSON expects JSONable dict or list, not JSON strings
  warnings.warn("JSON expects JSONable dict or list, not JSON strings")


<IPython.core.display.JSON object>

## 3. Szabadidő tervező ágens

Először készítsünk egy ágensgyűjteményt, amely kezeli az A2A által definiált ágenseinket!

In [40]:
agent_urls = ['http://localhost:10020', 'http://localhost:10021']

Az `AgentRegistry` feladata, hogy kezelje az A2A klienseket, amelyeken keresztül kommunikálhat a tervező ágensünk a fent definiált ágensekkel.

In [41]:
class AgentRegistry:
    agent_client_map: Dict[str, A2AClient] = {}
    agent_card_map: Dict[str, AgentCard] = {}

    def __init__(self, agent_urls: List[str]):
        self.agent_urls = agent_urls

    async def _create_client(self, url: str):
        async with httpx.AsyncClient() as httpx_client:
            resolver = A2ACardResolver(
                httpx_client=httpx_client,
                base_url=url,
            )

            public_card = (
                await resolver.get_agent_card()
            )


        self.agent_card_map[public_card.name] = public_card

        async_httpx_client = httpx.AsyncClient(timeout=httpx.Timeout(120.0))
        client = A2AClient(
            httpx_client=async_httpx_client,
            agent_card=public_card,
        )
        self.agent_client_map[public_card.name] = client

    async def create_clients(self):
        for url in self.agent_urls:
            await self._create_client(url)

In [42]:
registry = AgentRegistry(agent_urls)
await registry.create_clients()

INFO:     127.0.0.1:34534 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:53076 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


/tmp/ipykernel_11679/1378191031.py:23: DeprecationWarning: A2AClient is deprecated and will be removed in a future version. Use ClientFactory to create a client with a JSON-RPC transport.
  client = A2AClient(


In [51]:
class ProgramPlannerAgent:
    def __init__(self, registry: AgentRegistry):
        self.registry = registry
        self.agent = self._create_agent()

    def _agent_description(self, agent_name: str):
        agent_card = self.registry.agent_card_map[agent_name]
        return (
            f"Agent name: {agent_card.name}\n"
            f"Agent description: {agent_card.description}\n"
            f"Agent skills: {', '.join([f'{skill.name} (examples: {', '.join(skill.examples)})' for skill in agent_card.skills])}\n"
        )

    def system_instruction(self):
        return (
            "You are a free-time program planning assistant whose job is to plan free-time activities based on user requests.\n"
            "Directives:\n"
            ""
            "- Understand the user's request and break it down into smaller tasks.\n"
            "- Don't come up with program recommendations on your own; always use other agents you know to recommend programs.\n"
            "- When organizing an event, communicate only with agents you know.\n"
            "- Use the `send_message` tool to send a message to the agent you want to communicate with.\n"
            "- Don't forget to add the suggested event to the user's calendar as well.\n"
            "- If there’s an agent that can tell you what day it is today, ask it first instead of guessing on your own.\n"
            "- If something isn't clear, don't try to figure it out on your own, ask the user for clarification.\n"
            "- Use specific dates instead of phrases like 1today' or 'tomorrow' if you know the exact dates."
            "\n===Agents===\n"
            f"{'\n'.join([self._agent_description(agent) for agent in self.registry.agent_card_map.keys()])}"
        )

    async def send_message(self, agent_name: str, message: str):
      """Send a message to another agent."""
      agent_client = self.registry.agent_client_map[agent_name]
      if not agent_client:
        return "The given agent does not exist. Choose an existing one."


      parts = [Part(text=message)]
      message = Message(
          role=Role.user,
          parts=parts,
          message_id=uuid4().hex,
      )

      try:
        response = await agent_client.send_message(SendMessageRequest(id=uuid4().hex, params=MessageSendParams(message=message)))
        return response.root.result.artifacts

      except Exception:
        return "The given agent is not available currently."

      return ""

    def _create_agent(self):
        return Agent(
            model_name,
            system_prompt=self.system_instruction(),
            tools=[self.send_message]
        )

    async def run(self, message: str):
        response = await self.agent.run(message)
        return response.output

In [52]:
program_planner_agent = ProgramPlannerAgent(registry)

/usr/local/lib/python3.12/dist-packages/pydantic_ai/agent/__init__.py:409: PydanticAIDeprecationWarning: In v2.0, 'openai:' will resolve to the OpenAI Responses API by default. Use 'openai-chat:' to keep current Chat Completions behavior, or 'openai-responses:' to opt in early.
  self._model = models.infer_model(model)


In [53]:
result = await program_planner_agent.run("Pick an adventure movie for me that I can watch in the evening on today!")

15:45:22.086 agent run
15:45:22.091   chat gpt-4o-mini
15:45:23.305   running tool: send_message
15:45:23.305     a2a.client.transports.jsonrpc.JsonRpcTransport.send_message
15:45:23.306       a2a.client.transports.jsonrpc.JsonRpcTransport._get_http_args
15:45:23.306       a2a.client.transports.jsonrpc.JsonRpcTransport._apply_interceptors
15:45:23.307       a2a.client.transports.jsonrpc.JsonRpcTransport._send_request
15:45:23.311 a2a.server.request_handlers.jsonrpc_handler.JSONRPCHandler.on_message_send
15:45:23.312   a2a.server.request_handlers.default_request_handler.DefaultRequestHandler.on_message_send
15:45:23.312     a2a.server.request_handlers.default_request_handler.DefaultRequestHandler._setup_message_execution
15:45:23.312       a2a.server.events.in_memory_queue_manager.InMemoryQueueManager.create_or_tap
15:45:23.313       a2a.server.request_handlers.default_request_handler.DefaultRequestHandler._register_producer
15:45:23.314     a2a.server.events.event_consumer.EventConsume

In [54]:
display(Markdown(result))

I've added a screening of **"Interstellar"** to your calendar for **May 19, 2026**, starting at **8:00 PM**.

Additionally, you can consider watching another movie at the same time:

1. **The Dark Knight**
   - **Genre:** Action, Thriller
   - **Description:** When the menace known as the Joker wreaks havoc and chaos on the people of Gotham.
   - **Length:** 152 minutes
   - **Screening Time:** 9:00 PM

Let me know if you need any more information or assistance!

### Ellenőrzés

Egy egyszerű lekérdezéssel ellenőrizzük, hogy valóban megjelent a megfelelő bejegyzéssel a naptárban az esemény.

In [55]:
def get_events_after_date(target_date: datetime, days_offset: int = 0):
  selected_date = target_date + timedelta(days=days_offset)
  selected_date = selected_date.replace(hour=0, minute=0, second=0, microsecond=0)

  with Session(calendar_engine) as session:
    statement = select(CalendarEntry).where(
      CalendarEntry.start_time >= selected_date,
      CalendarEntry.start_time < selected_date + timedelta(days=1)
    )
    results = session.exec(statement).all()
    return results

In [56]:
get_events_after_date(datetime.now(), 0)

[CalendarEntry(title='Screening of Interstellar', start_time=datetime.datetime(2026, 5, 19, 20, 0), description='Watch the movie Interstellar', end_time=datetime.datetime(2026, 5, 19, 23, 0), id=1)]